In [1]:
!pip -q install boto3 requests python-dotenv



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Natha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()  # loads .env into environment

True

In [3]:
import os
import requests
import boto3
from urllib.parse import urlparse

def upload_listenbrainz_dump_to_s3(
    url: str,
    bucket: str,
    prefix: str = "listenbrainz/",
    s3_key: str | None = None,
    region: str | None = None,
    timeout: int = 60,
):
    """
    Streams a remote ListenBrainz dump file (HTTP/HTTPS) directly into S3.

    - No full download to local disk
    - Uses multipart upload via boto3.upload_fileobj for large files
    """

    # Pick filename from URL if key not explicitly provided
    if s3_key is None:
        path = urlparse(url).path
        filename = os.path.basename(path.rstrip("/"))
        if not filename:
            raise ValueError("Could not infer filename from URL. Provide s3_key explicitly.")
        s3_key = f"{prefix.rstrip('/')}/{filename}"

    # Create S3 client (boto3 reads creds from env / AWS config automatically)
    session = boto3.session.Session(region_name=region or os.getenv("AWS_DEFAULT_REGION"))
    s3 = session.client("s3")

    # Stream the download
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()

        # Optional: show size if server provides it
        size = r.headers.get("Content-Length")
        if size:
            print(f"Remote file size: {int(size):,} bytes")

        print(f"Uploading to s3://{bucket}/{s3_key} ...")

        # upload_fileobj will do multipart uploads automatically for large streams
        s3.upload_fileobj(
            Fileobj=r.raw,
            Bucket=bucket,
            Key=s3_key,
            ExtraArgs={
                # good practice: preserve content-type if present
                "ContentType": r.headers.get("Content-Type", "application/octet-stream"),
                # optional: set cache control / metadata if you want
            },
        )

    print("Done.")
    return f"s3://{bucket}/{s3_key}"


In [5]:
#url = "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2420-20260205-000003-incremental/listenbrainz-listens-dump-2420-20260205-000003-incremental.tar.zst"

urls = [
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2430-20260215-000003-incremental/listenbrainz-listens-dump-2430-20260215-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2429-20260214-000003-incremental/listenbrainz-listens-dump-2429-20260214-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2428-20260213-000003-incremental/listenbrainz-listens-dump-2428-20260213-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2427-20260212-000003-incremental/listenbrainz-listens-dump-2427-20260212-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2426-20260211-000003-incremental/listenbrainz-listens-dump-2426-20260211-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2425-20260210-000003-incremental/listenbrainz-listens-dump-2425-20260210-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2424-20260209-000003-incremental/listenbrainz-listens-dump-2424-20260209-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2423-20260208-000003-incremental/listenbrainz-listens-dump-2423-20260208-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2422-20260207-000003-incremental/listenbrainz-listens-dump-2422-20260207-000003-incremental.tar.zst",
    "https://ftp.musicbrainz.org/pub/musicbrainz/listenbrainz/incremental/listenbrainz-dump-2421-20260206-000003-incremental/listenbrainz-listens-dump-2421-20260206-000003-incremental.tar.zst",

]
bucket = "stall-munezero-final-project"
prefix = "listenbrainz/incremental/"

for url in urls:
    s3_uri = upload_listenbrainz_dump_to_s3(url=url, bucket=bucket, prefix=prefix)

s3_uri


Remote file size: 148,860,199 bytes
Uploading to s3://stall-munezero-final-project/listenbrainz/incremental/listenbrainz-listens-dump-2430-20260215-000003-incremental.tar.zst ...
Done.
Remote file size: 148,845,218 bytes
Uploading to s3://stall-munezero-final-project/listenbrainz/incremental/listenbrainz-listens-dump-2429-20260214-000003-incremental.tar.zst ...
Done.
Remote file size: 162,461,461 bytes
Uploading to s3://stall-munezero-final-project/listenbrainz/incremental/listenbrainz-listens-dump-2428-20260213-000003-incremental.tar.zst ...
Done.
Remote file size: 124,180,014 bytes
Uploading to s3://stall-munezero-final-project/listenbrainz/incremental/listenbrainz-listens-dump-2427-20260212-000003-incremental.tar.zst ...
Done.
Remote file size: 194,735,478 bytes
Uploading to s3://stall-munezero-final-project/listenbrainz/incremental/listenbrainz-listens-dump-2426-20260211-000003-incremental.tar.zst ...
Done.
Remote file size: 142,418,198 bytes
Uploading to s3://stall-munezero-final-

's3://stall-munezero-final-project/listenbrainz/incremental/listenbrainz-listens-dump-2421-20260206-000003-incremental.tar.zst'